# VoiceGuard — train on Kaggle (P100)

Full pipeline: HF data → content-matched MMS-TTS fakes → manifests → RawBoost →
fine-tune wav2vec2 + AASIST end to end → evaluate per dataset/language.
Output: `/kaggle/working/aasist_indicw2v.pt` — download from the Output tab, drop into
`backend/models/` locally.

**Settings → Accelerator → GPU P100** (needs one-time phone verification).
**Save Version → Save & Run All (Commit)** runs in the background past the 12 h limit.
Data + checkpoints live in `/kaggle/working` (kept across commits), so if a session is
killed, just **run a new version** — training resumes from the last epoch.

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'Settings → Accelerator → GPU'
print(torch.cuda.get_device_name(0))
!pip -q install -U 'transformers>=4.44' 'datasets>=2.20' huggingface_hub soundfile librosa pyyaml scipy

In [ ]:
REPO_URL = 'https://github.com/Deva-996/VoiceGuard'
HF_TOKEN = ''   # optional: enables ai4bharat/indicwav2vec-hindi (accept its licence first)

%cd /kaggle/working
!rm -rf voiceguard && git clone --depth 1 {REPO_URL} voiceguard
%cd voiceguard
os.makedirs('/kaggle/working/vgdata', exist_ok=True)
!rm -rf data && ln -s /kaggle/working/vgdata data
if HF_TOKEN: os.environ['HF_TOKEN'] = HF_TOKEN
!du -sh /kaggle/working/vgdata 2>/dev/null; ls

In [ ]:
FRONTEND = 'facebook/wav2vec2-xls-r-300m'   # or ai4bharat/indicwav2vec-hindi (needs HF_TOKEN)
EPOCHS   = 16
UNFREEZE = 5        # frontend frozen for N epochs, then fine-tuned (the quality lever)
LOSS     = 'oc_softmax'
FAKE_N   = 1200
LIMIT    = None     # e.g. 400 for a smoke run first

import yaml, pathlib
p = pathlib.Path('training/config_train.yaml'); cfg = yaml.safe_load(p.read_text())
cfg['device'] = 'cuda'
cfg['frontend'].update(model_id=FRONTEND, layer=-1, stage2_unfreeze_epoch=UNFREEZE)
cfg['loss']['name'] = LOSS
cfg['epochs'] = EPOCHS
cfg['num_workers'] = 2
cfg['checkpoint']['out'] = '/kaggle/working/aasist_indicw2v.pt'
p.write_text(yaml.safe_dump(cfg, sort_keys=False)); print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
args = f'--config training/config_train.yaml --fake-n {FAKE_N} --epochs {EPOCHS}'
if LIMIT: args += f' --limit {LIMIT}'
# data persists in /kaggle/working -> skip re-download/re-fake if already there
if os.path.exists('data/manifests/train.tsv'): args += ' --skip-download --skip-fakes'
!python -m training.pipeline_run {args}

In [ ]:
import torch
c = torch.load('/kaggle/working/aasist_indicw2v.pt', map_location='cpu', weights_only=False)
print('dev_eer', c.get('dev_eer'), '| epoch', c.get('epoch'), '| frontend', c.get('frontend_model_id'),
      '| finetuned', c.get('frontend') is not None)
!ls -la /kaggle/working/*.pt